# Fish Toxicity Regression
In this dataset, each row is an experiment on the rainbow trout. The columns show what chemicals were present, how long the fish were left in the toxic water, the temperature of the water, etc. With that, our target variable is "result_conc1_mean_mol_log" which is the The base-10 logarithm of the mean molar concentration log10(mol/L) required to reach the specified experimental toxicity endpoint (LC50)

In this version:
- pick features to include based on these categories: Test conditions, molecule description, and chemical properties
- change back into a regression task (idea of first having a regression task, picking a threshold, and classifying whether toxic or non-toxic)
- scaling features

In [79]:
import pandas as pd
import numpy as np
from sklearn.model_selection import PredefinedSplit

# read csv
# the fingerprints must be read as text, otherwise pandas turns the 0/1 strings into huge numbers
fp_text = {col: str for col in ["chem_MACCS_fp", "chem_pcp_fp", "chem_Morgan_fp", "chem_ToxPrint_fp"]}
df = pd.read_csv(
    "../adore_dataset/processed/s-F2F-1_mortality.csv",
    dtype=fp_text,
    low_memory=False,
)

target = "result_conc1_mean_mol_log"

# test conditions (decided before the experiment, so no leakage)
categorical_cols = ["test_exposure_type", "result_conc1_type", "test_media_type"]
ordinal_cols = ["result_obs_duration_mean"]          # 24/48/72/96 h

# chemical properties
numeric_cols = ["chem_mw", "chem_ws", "chem_mp", "chem_rdkit_clogp"]

# molecule description: MACCS fingerprint, one column per bit
maccs = df["chem_MACCS_fp"].apply(lambda s: pd.Series(list(s))).astype(int)
maccs.columns = [f"MACCS_{i:03d}" for i in maccs.columns]

X = pd.concat([df[categorical_cols + ordinal_cols + numeric_cols], maccs], axis=1)
y = df[target]

# train/test split: the predefined split by chemical (occurrence)
train = df["split_occurrence"] != "test"
test = df["split_occurrence"] == "test"

X_train, X_test = X[train], X[test]
y_train, y_test = y[train], y[test]

# keep only bits that vary (not always 1s or not always 0s)
fp_cols = [c for c in maccs.columns if X_train[c].std() >= 0.1]
X_train = X_train[categorical_cols + ordinal_cols + numeric_cols + fp_cols]
X_test = X_test[categorical_cols + ordinal_cols + numeric_cols + fp_cols]

# cross-validation folds 0-4 from the same predefined split
ps = PredefinedSplit(df.loc[train, "split_occurrence"].astype(int).values)

print("train:", X_train.shape, " test:", X_test.shape, " MACCS bits kept:", len(fp_cols))
print("missing values:", X_train.isna().sum().sum(), X_test.isna().sum().sum())


train: (3928, 151)  test: (713, 151)  MACCS bits kept: 143
missing values: 0 0


## Preprocessing

In [80]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

preprocessing = ColumnTransformer([
    ("scaler", StandardScaler(), numeric_cols),
    ("one_hot", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
    ("duration", MinMaxScaler(), ordinal_cols),       
    ("maccs", "passthrough", fp_cols),               
])

In [81]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_validate
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

## Cross-validation
RMSE is in log10 units, so an RMSE of 1 means predictions are off by about one order of magnitude on the LC50 scale.

In [82]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Elastic Net": ElasticNet(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

cv_results = []
for name, model in models.items():
    pipeline = make_pipeline(preprocessing, model)
    scores = cross_validate(pipeline, X_train, y_train, cv=ps,
                            scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"})
    rmse = -scores["test_rmse"]
    cv_results.append({"model": name,
                       "RMSE": rmse.mean(), "RMSE std": rmse.std(),
                       "R2": scores["test_r2"].mean()})
    print(f"{name:18s} RMSE {rmse.mean():.3f} ± {rmse.std():.3f}   R2 {scores['test_r2'].mean():.3f}")


Linear Regression  RMSE 1.244 ± 0.174   R2 0.372
Ridge              RMSE 1.199 ± 0.163   R2 0.418
Lasso              RMSE 1.579 ± 0.132   R2 0.012
Elastic Net        RMSE 1.352 ± 0.110   R2 0.276
KNN                RMSE 1.197 ± 0.264   R2 0.395
SVR                RMSE 0.966 ± 0.129   R2 0.621
Decision Tree      RMSE 1.630 ± 0.129   R2 -0.103
Random Forest      RMSE 1.089 ± 0.126   R2 0.506
Gradient Boosting  RMSE 1.056 ± 0.096   R2 0.545


## Test set
Only the model with the best cross-validation RMSE is evaluated on the test set, once only.

In [83]:
# retrain the best model (lowest CV RMSE) on all training data and evaluate once on the test set
best_name = "SVR"
best_pipeline = make_pipeline(preprocessing, models[best_name]).fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

print("best model:", best_name)
print(f"test RMSE: {root_mean_squared_error(y_test, y_pred):.3f}")
print(f"test MAE:  {mean_absolute_error(y_test, y_pred):.3f}")
print(f"test R2:   {r2_score(y_test, y_pred):.3f}")

best model: SVR
test RMSE: 0.940
test MAE:  0.709
test R2:   0.645
